In [1]:
import os
import glob
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import median_filter, label as nd_label, binary_dilation
import cv2

input_folder = r"fits_downloads"
output_folder = r"recreate"
fixed_output_size = 224

os.makedirs(output_folder, exist_ok=True)

In [2]:
def get_background(img):
    h, w = img.shape
    edge = min(40, h//8, w//8)
    
    corners = np.concatenate([
        img[:edge, :edge].flatten(),
        img[:edge, -edge:].flatten(),
        img[-edge:, :edge].flatten(),
        img[-edge:, -edge:].flatten()
    ])
    
    _, median, std = sigma_clipped_stats(corners, sigma=2.5, maxiters=5)
    return median, std

In [3]:
def enhance_galaxy(img, bg, noise):
    cleaned = img - bg
    cleaned[cleaned < 0] = 0
    
    smooth1 = median_filter(cleaned, size=3)
    smooth2 = median_filter(cleaned, size=5)
    enhanced = (smooth1 + smooth2) / 2
    
    return enhanced

In [4]:
def find_central_object(img, bg, noise, search_radius=None):
    cy, cx = img.shape[0] // 2, img.shape[1] // 2
    
    enhanced = enhance_galaxy(img, bg, noise)
    
    threshold = max(2.5 * noise, np.percentile(enhanced, 75))
    mask = enhanced > threshold
    
    if not np.any(mask):
        return 50
    
    mask = binary_dilation(mask, iterations=2)
    labels, n = nd_label(mask)
    
    if n == 0:
        return 5
    best_label = 0
    best_score = -1
    
    for i in range(1, n + 1):
        region = (labels == i)
        size = np.sum(region)
        
        if size < 50:
            continue
    
        y_coords, x_coords = np.where(region)
        reg_cy, reg_cx = np.mean(y_coords), np.mean(x_coords)
        
        dist = np.sqrt((reg_cx - cx)**2 + (reg_cy - cy)**2)
        
        brightness = np.sum(enhanced[region])
        
        score = brightness / (1 + dist/10)
        
        if score > best_score:
            best_score = score
            best_label = i
    
    if best_label == 0:
        return 50
    
    mask = (labels == best_label)
    y, x = np.where(mask)
    
    distances = np.sqrt((x - cx)**2 + (y - cy)**2)
    radius = int(np.percentile(distances, 90))
    
    return max(30, min(radius, min(img.shape) // 2 - 10))

In [5]:
def resize_img(img, target_size):
    if img.shape[0] > target_size:
        return cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
    else:
        return cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_CUBIC)

In [6]:
fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
print(f"Found {len(fits_files)} files\n")

Found 584 files



In [7]:
for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    print(f"[{i}/{len(fits_files)}] {fname}")
    
    try:
        with fits.open(fpath) as hdul:
            img = hdul[0].data
            hdr = hdul[0].header
            bg, noise = get_background(img)
            print(f"  BG: {bg:.4f}, Noise: {noise:.4f}")
            
            radius = find_central_object(img, bg, noise)
            print(f"  Radius: {radius}px")
            
            size = int(radius * 2 * 1.4)  # more padding
            size += size % 2
            size = min(size, min(img.shape))
            
            pos = (img.shape[1] // 2, img.shape[0] // 2)
            cut = Cutout2D(img, pos, size, wcs=WCS(hdr))
            
            new_hdr = hdr.copy()
            new_hdr.update(cut.wcs.to_header())
            
            fits.PrimaryHDU(cut.data, new_hdr).writeto(
                os.path.join(output_folder, f"cutout_{fname}"), overwrite=True)
            
            resized = resize_img(cut.data.astype(np.float32), fixed_output_size)
            fits.PrimaryHDU(resized, new_hdr).writeto(
                os.path.join(output_folder, f"resized_{fname}"), overwrite=True)
            
            print(f"  Saved: {size}x{size} -> {fixed_output_size}x{fixed_output_size}")
            
            if i % 10 == 0:
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                
                v1, v2 = np.percentile(img, [1, 99])
                axes[0].imshow(img, origin='lower', cmap='gray', vmin=v1, vmax=v2)
                axes[0].set_title(f'{fname}\n{img.shape[0]}x{img.shape[1]}')
                axes[0].plot(pos[0], pos[1], 'r+', markersize=15, markeredgewidth=2)
                axes[0].add_patch(plt.Circle(pos, radius, color='r', fill=False, ls='--', lw=2))
                
                v1, v2 = np.percentile(cut.data, [1, 99])
                axes[1].imshow(cut.data, origin='lower', cmap='gray', vmin=v1, vmax=v2)
                axes[1].set_title(f'Cutout: {size}x{size}')
                
                v1, v2 = np.percentile(resized, [1, 99])
                axes[2].imshow(resized, origin='lower', cmap='gray', vmin=v1, vmax=v2)
                axes[2].set_title(f'{fixed_output_size}x{fixed_output_size}')
                
                plt.tight_layout()
                plt.savefig(os.path.join(output_folder, f"comp_{os.path.splitext(fname)[0]}.png"), 
                           dpi=150, bbox_inches='tight')
                plt.close()
                print(f"  → Plot saved for image #{i}: {fname}")
            
    except Exception as e:
        print(f"  Error: {e}")

print("\nDone!")

[1/584] ARP_314_NED01.fits
  BG: 0.0031, Noise: 0.0015
  Radius: 30px
  Saved: 84x84 -> 224x224
[2/584] ARP_314_NED02.fits
  BG: 0.0031, Noise: 0.0015
  Radius: 64px
  Saved: 180x180 -> 224x224
[3/584] Centaurus_A.fits
  BG: 0.0089, Noise: 0.0009
  Radius: 115px
  Saved: 300x300 -> 224x224
[4/584] CGCG_023-019.fits
  BG: 0.0040, Noise: 0.0014
  Radius: 30px
  Saved: 84x84 -> 224x224
[5/584] CGCG_097-068.fits
  BG: 0.0025, Noise: 0.0010
  Radius: 33px
  Saved: 92x92 -> 224x224
[6/584] CGCG_377-039.fits
  BG: 0.0031, Noise: 0.0003
  Radius: 140px
  Saved: 300x300 -> 224x224
[7/584] ESO_034-G013.fits
  BG: 0.0049, Noise: 0.0028
  Radius: 30px
  Saved: 84x84 -> 224x224
[8/584] ESO_204-G006.fits
  BG: 0.0027, Noise: 0.0004
  Radius: 30px
  Saved: 84x84 -> 224x224
[9/584] ESO_243-G051.fits
  BG: 0.0023, Noise: 0.0005
  Radius: 30px
  Saved: 84x84 -> 224x224
[10/584] ESO_291-G009.fits
  BG: 0.0022, Noise: 0.0008
  Radius: 30px
  Saved: 84x84 -> 224x224
  → Plot saved for image #10: ESO_291-G0